# Simulation-based calibration and coverage

`axiom.diagnose` is the trust machinery: before a fitted surface is allowed to answer a
decision question, the *procedure* that produced it must be shown to work on worlds where the
truth is known. This notebook covers the two procedure-level checks:

- **SBC** (Talts et al. 2018): draw parameters from the prior, simulate an outcome, refit, and
  rank the true value among the posterior draws. If the fit is calibrated, the ranks are
  uniform. `sbc` works on any `ModelSpec`; `sbc_surface` and `sbc_pool` wrap the surface and
  meta models and simulate through the *same* `forward` the likelihood uses.
- **Coverage**: fix a truth, replicate the world with new noise, fit, and count how often the
  stated interval covers the truth. Every rate comes with the exact Clopper–Pearson region for
  its `N`, so "90 % of 10" is judged against the region for `n = 10`, not against 0.9.

Everything here uses the Laplace backend and tiny worlds so the notebook runs in seconds; the
`slow` tier of the contract tests runs the same checks at `N ≥ 200`.

In [ ]:
import numpy as np

from axiom.core import (
    Add, Data, Likelihood, ModelSpec, Mul, Param, Population, Prior, Spec, TimeWindow,
    clopper_pearson, dimensionless,
)
from axiom.diagnose import (
    CoverageResult, EstimandCoverage, EstimandCoverageResult, Fitter, ParameterCoverage,
    ParameterRanks, SBCResult, SBCSpec, Simulator, SupportsWorld, WorldFactory, WorldView,
    coverage, default_bins, draw_prior, estimand_coverage, misspecify, rank_uniformity, sbc,
    sbc_pool, sbc_surface, simulate_outcome, truth_producer,
)
from axiom.build import MetaBuilder
from axiom.estimands import Level, realize, standard_estimands
from axiom.meta import PoolPriors, PoolSpec
from axiom.sim import DosePlan, surface_world
from axiom.surface import FitResult, GeometricCarryover, HillKernel, LinearKernel, NoCarryover, fit

from axiom.display import enable

enable();  # every axiom result renders itself from here on

DL = dimensionless()
AMP = Prior(family="lognormal", hyper={"mu": 0.0, "sigma": 0.5})

## Prior draws and simulated outcomes

`draw_prior` samples every free parameter of a `ModelSpec` from its prior, resolving
hierarchical hyper-parameters in dependency order; `simulate_outcome` draws the outcome from the
model's likelihood family at the mean implied by one parameter draw. Together they are the
default `Simulator` that `sbc` uses; you may pass your own with the same
`(theta, rng) -> outcome` signature.

In [ ]:
def line(beta_mu: float = 0.0, beta_sd: float = 1.0) -> ModelSpec:
    alpha = Param(name="alpha", dimension=DL, prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 1.0}))
    beta = Param(name="beta", dimension=DL, prior=Prior(family="normal", hyper={"mu": beta_mu, "sigma": beta_sd}))
    sigma = Param(name="sigma", dimension=DL, prior=Prior(family="fixed", hyper={"value": 0.5}))
    return ModelSpec(
        name="line",
        mean=Add(terms=(alpha, Mul(factors=(beta, Data(name="x", dimension=DL))))),
        outcome=Data(name="y", dimension=DL),
        likelihood=Likelihood(family="normal", scale="sigma"),
        parameters=(alpha, beta, sigma),
    )


model = line()
data = {"x": np.linspace(-1.0, 1.0, 30), "y": np.zeros(30)}
rng = np.random.default_rng(0)
theta = draw_prior(model, rng, n=3)
print({k: np.round(v, 3) for k, v in theta.items()})
one = {k: v[0] for k, v in theta.items()}
y = simulate_outcome(model, data, one, rng)
print("simulated outcome (first 5):", np.round(y[:5], 3))


def my_simulator(theta: dict[str, np.ndarray], rng: np.random.Generator) -> np.ndarray:
    return simulate_outcome(model, data, theta, rng)


simulate: Simulator = my_simulator

## Rank statistics

`rank_uniformity` is the test applied to each parameter's ranks: a chi-square test over
`bins` equiprobable bins (`default_bins` picks the largest divisor of `n_ranks + 1` that keeps
at least ten expected counts per bin) and an ECDF-difference statistic against a simultaneous
band. The result is a `ParameterRanks` spec that round-trips with its histogram.

In [ ]:
print("default bins for N=60, L=19:", default_bins(60, 19), "| N=200:", default_bins(200, 19))
uniform = rank_uniformity("u", rng.integers(0, 20, size=200), n_ranks=19, alpha=0.05)
skewed = rank_uniformity("s", np.minimum(rng.integers(0, 8, size=200), 19), n_ranks=19, alpha=0.05)
for r in (uniform, skewed):
    assert isinstance(r, ParameterRanks)
    print(f"{r.name}: bins={r.bins} chi2 p={r.chi2_p_value:.3g} ecdf={r.ecdf_statistic:.3f} "
          f"band={r.ecdf_band:.3f} passed={r.passed}")
print("histogram:", uniform.histogram)

## `sbc` on a model whose Laplace posterior is exact

`y = alpha + beta·x` with a fixed noise sd has a Gaussian posterior, so the Laplace fit is the
exact posterior and the ranks must be uniform. `SBCSpec` fixes the number of simulations, the
draws per refit, the rank resolution `rank_draws`, the backend, and the alpha; the alpha is
split evenly across the ranked parameters (`alpha_per_parameter`). Failed refits are counted,
never dropped.

In [ ]:
spec = SBCSpec(n_simulations=24, draws=60, rank_draws=19, seed=3, alpha=0.05)
out = sbc(model, data, spec=spec, simulate=simulate)
assert isinstance(out, SBCResult)
print(f"fitted {out.n_fitted}/{out.n_simulations}, failed {out.n_failed_fits}, "
      f"alpha per parameter {out.alpha_per_parameter:.4f}, passed={out.passed}")
for p in out.parameters:
    print(f"  {p.name:6s} histogram={p.histogram} chi2 p={p.chi2_p_value:.3f} passed={p.passed}")
print("round-trips:", SBCResult.from_json(out.to_json()) == out)

### Negative control

Refitting with a *different* prior on `beta` (shifted to 3 and narrowed) is the classic
mistake SBC exists to catch: the ranks pile up at one end and the check fails. Both hashes
are recorded, so the result says which model simulated and which refit.

In [ ]:
wrong = sbc(model, data, spec=spec.model_copy(update={"parameters": ("beta",)}), simulate=simulate, refit=line(3.0, 0.1))
assert isinstance(wrong, SBCResult)
print("refit hash differs:", wrong.refit_model_hash != wrong.model_hash)
print("beta histogram:", wrong.parameters[0].histogram, "| failed:", wrong.failed_parameters)

## `sbc_surface` and `sbc_pool`

The surface variant simulates the outcome through `Surface.forward` — the one `forward` —
at prior-drawn parameters on a template panel; the pool variant does the same for the meta
model over a corpus. Both return the same `SBCResult`.

In [ ]:
world = surface_world(
    n_units=3, n_periods=6, treatments=("a",), kernels=LinearKernel(reference_dose=1.0, amplitude_prior=AMP),
    intercept="shared", noise_sd=0.3, seed=1, doses=DosePlan(zero_fraction=0.2),
)
surf = sbc_surface(world.spec, world.panel, sbc_spec=SBCSpec(n_simulations=6, draws=40))
assert isinstance(surf, SBCResult)
print("surface parameters ranked:", [p.name for p in surf.parameters], "| fitted:", surf.n_fitted)

corpus = MetaBuilder().name("demo").family("f")
for i, (y_i, se_i) in enumerate(zip([0.42, 0.55, 0.31, 0.67, 0.48, 0.39, 0.52, 0.44], [0.1, 0.15, 0.12, 0.2, 0.11, 0.14, 0.13, 0.1])):
    corpus = corpus.study(f"s{i}", estimate=y_i, se=se_i, read="experiment", contributor=f"c{i % 3}")
pooled = sbc_pool(PoolSpec(family="f", priors=PoolPriors(tau_fixed=0.3)), corpus.build(), sbc_spec=SBCSpec(n_simulations=6, draws=40))
assert isinstance(pooled, SBCResult)
print("pool parameters ranked:", [p.name for p in pooled.parameters], "| passed:", pooled.passed)

## Coverage of a fixed truth

`coverage` takes a `WorldFactory` — any `seed -> SupportsWorld` (something with `spec`,
`panel`, and the true `theta`) — and refits `n` replications. Each parameter gets a
`ParameterCoverage` row: covered count, the stated interval definition and mass, and the exact
acceptance region for `(n, mass, alpha)`. Pass a custom `Fitter` to change how each world is
fitted.

In [ ]:
def factory(seed: int) -> SupportsWorld:
    return surface_world(
        n_units=3, n_periods=6, treatments=("a",), kernels=LinearKernel(reference_dose=1.0, amplitude_prior=AMP),
        intercept="shared", noise_sd=0.3, seed=seed, doses=DosePlan(zero_fraction=0.2),
    )


make_world: WorldFactory = factory


def fitter(world: SupportsWorld, seed: int) -> FitResult:
    return fit(world.spec, world.panel, backend="laplace", draws=60, chains=1, seed=seed)


fit_one: Fitter = fitter
cov = coverage(make_world, n=10, mass=0.9, definition="eti", fit=fit_one, seed=5, alpha=0.01)
assert isinstance(cov, CoverageResult)
print("exact region for n=10, p=0.9, alpha=0.01:", clopper_pearson(10, 0.9, 0.01))
for p in cov.parameters:
    assert isinstance(p, ParameterCoverage)
    print(f"  {p.name:12s} covered {p.covered}/{p.n} region=[{p.region.lower}, {p.region.upper}] passed={p.passed}")
print("passed:", cov.passed, "| round-trips:", CoverageResult.from_json(cov.to_json()) == cov)

## Estimand coverage and misspecified views

`truth_producer` turns a world into a zero-variance producer so any `Estimand` can be realized
at the truth; `estimand_coverage` then scores realized intervals against those true values.
`misspecify` wraps a world in a `WorldView` that keeps the panel and truth but fits a *different*
spec — here a carryover world fitted without carryover — so the coverage of a wrong model can be
measured against the true world's estimand.

In [ ]:
def hill_world(seed: int) -> SupportsWorld:
    return surface_world(
        n_units=3, n_periods=6, treatments=("a",), kernels=HillKernel(reference_dose=1.0, amplitude_prior=AMP),
        intercept="shared", noise_sd=0.3, seed=seed, doses=DosePlan(zero_fraction=0.2),
    )


w = hill_world(0)
producer = truth_producer(w)
est = standard_estimands(
    w.spec.treatments[0], w.spec.outcome, Population(name="all"), TimeWindow(start=0, stop=6),
    Level(unit="aggregate"), dose=1.0,
)
contrast = est.get("contrast_at_dose")
truth = realize(contrast, producer, assume_identified=True, definition="eti", mass=0.9)
print("true contrast at dose 1:", round(truth.summary.mean, 4), "(sd", truth.summary.sd, ")")

ec = estimand_coverage(hill_world, estimands=[contrast], n=6, draws=60, seed=1)
assert isinstance(ec, EstimandCoverageResult)
for e in ec.estimands:
    assert isinstance(e, EstimandCoverage)
    print(f"  {e.name}: covered {e.covered}/{e.n}, true values {np.round(e.true_values, 2)}, passed={e.passed}")

In [ ]:
carry = surface_world(
    n_units=3, n_periods=6, treatments=("a",), kernels=HillKernel(reference_dose=1.0, amplitude_prior=AMP),
    carryover=GeometricCarryover(max_lag=3), intercept="shared", noise_sd=0.3, seed=0,
)
naive_spec = carry.spec.model_copy(update={"carryover": {"a": NoCarryover()}})
view = misspecify(carry, naive_spec)
assert isinstance(view, WorldView) and isinstance(view, SupportsWorld)
print("view fits", view.spec.carried, "carryover but the truth still has", sorted(view.theta)[:3], "...")
res = fit_one(view, 0)
print("fitted the naive spec:", res.converged, "| free parameters:", [p.name for p in res.surface.model.free])